# Maestra Grupo B - Problema 04 FE historico

Correr en orden: celda 1 (configurar), 2 (verificar), 3 (ejecutar), 4 (consolidar).

In [8]:
library(data.table)
library(yaml)

# ==========================================
# NOTEBOOK MAESTRA - GRUPO B
# ==========================================

# ---- LO UNICO QUE SE TOCA ----
CONFIGS    <- c("E5")   # luego: c("E0","E1","E2","E3","E4")
N_SEMILLAS <- 10         # 1 = primera corrida real (valida todo el circuito); luego 10

SEMILLAS <- c(
  770503, 216751, 126233, 877577, 388403,
  308563, 344785, 618973, 245039, 552707
)
semillas_a_correr <- SEMILLAS[seq_len(N_SEMILLAS)]

# Resultados: bucket persistente
BASE <- "/content/buckets/b1/exp/GrupoB_SinFE"
stopifnot(dir.exists("/content/buckets/b1"))
dir.create(file.path(BASE, "params"), recursive = TRUE, showWarnings = FALSE)

cat("Configuraciones:", CONFIGS, "\n")
cat("Semillas a correr:", semillas_a_correr, "\n")

Configuraciones: E5 
Semillas a correr: 770503 216751 126233 877577 388403 308563 344785 618973 245039 552707 


In [9]:
# ==========================================
# LOCALIZAR EL WORKFLOW Y VERIFICAR EL ENTORNO
# ==========================================

# carpeta P04_FEhistorico_GrupoB
CARPETA_NOTEBOOKS <- "/home/ds/labo2026v/P04_FEhistorico_GrupoB"

if (!nzchar(CARPETA_NOTEBOOKS)) {
  cand <- Sys.glob("/content/buckets/*/ExpColaborativos/P04_FEhistorico_GrupoB")
  if (length(cand) == 1) CARPETA_NOTEBOOKS <- cand
}
if (!nzchar(CARPETA_NOTEBOOKS))
  stop("No encontre la carpeta del workflow. Completar CARPETA_NOTEBOOKS a mano.")

WF_ORIGINAL  <- file.path(CARPETA_NOTEBOOKS, "02_WorkFlow_GrupoB_SinFE.ipynb")
WF_CONGELADO <- file.path(CARPETA_NOTEBOOKS, "02_WorkFlow_GrupoB_SinFE.CONGELADO.ipynb")
stopifnot(file.exists(WF_ORIGINAL))

# Se ejecuta una copia congelada: editar el workflow mientras corre no contamina las semillas.
# Si corregis algo despues de congelar, BORRA el congelado para que se regenere.
if (!file.exists(WF_CONGELADO)) {
  file.copy(WF_ORIGINAL, WF_CONGELADO)
  cat("Copia congelada creada:", WF_CONGELADO, "\n")
}
ARCHIVO_WORKFLOW <- WF_CONGELADO
cat("Workflow a ejecutar:", ARCHIVO_WORKFLOW, "\n")

# Jupyter y kernel de R
stopifnot(nzchar(Sys.which("jupyter")))
ks <- system2(Sys.which("jupyter"), c("kernelspec", "list"), stdout = TRUE)
print(ks)
stopifnot(any(grepl("^\\s*ir\\s", ks)))

Workflow a ejecutar: /home/ds/labo2026v/P04_FEhistorico_GrupoB/02_WorkFlow_GrupoB_SinFE.CONGELADO.ipynb 
[1] "Available kernels:"                                              
[2] "  python3       /home/ds/.venv/share/jupyter/kernels/python3"    
[3] "  ir            /home/ds/.local/share/jupyter/kernels/ir"        
[4] "  julia-1.10    /home/ds/.local/share/jupyter/kernels/julia-1.10"


In [10]:
# ==========================================
# EJECUCION: semilla afuera, configuracion adentro
# ==========================================

ARCHIVO_TIEMPOS <- file.path(BASE, "tiempos_corridas.csv")
fallos <- 0

for (semilla in semillas_a_correr) {
  for (cfg in CONFIGS) {

    id <- paste0(cfg, "_S", semilla)
    carpeta <- file.path(BASE, cfg, id)

    # ya completada (existe PARAM.yml): no repetir
    if (file.exists(file.path(carpeta, "PARAM.yml"))) {
      cat("Ya completada:", id, "\n"); next
    }

    # corrida incompleta de un intento anterior: limpiar para no mezclar archivos
    if (dir.exists(carpeta)) unlink(list.files(carpeta, full.names = TRUE), recursive = TRUE)
    dir.create(carpeta, recursive = TRUE, showWarnings = FALSE)

    archivo_param <- file.path(BASE, "params", paste0(id, ".rds"))
    saveRDS(list(
      configuracion      = cfg,
      semilla_primigenia = semilla,
      modo_prueba        = FALSE,
      id_ejecucion       = id
    ), archivo_param)

    Sys.setenv(GRUPOB_PARAMETROS = archivo_param)
    cat("\nINICIANDO:", id, format(Sys.time()), "\n")
    t0 <- Sys.time()

    estado <- system2(
      Sys.which("jupyter"),
      args = c("nbconvert", "--to", "notebook", "--execute",
               "--ExecutePreprocessor.timeout=-1",
               "--output", paste0(id, "_ejecutado.ipynb"),
               "--output-dir", shQuote(carpeta),
               shQuote(ARCHIVO_WORKFLOW)),
      stdout = file.path(carpeta, "salida.log"),
      stderr = file.path(carpeta, "errores.log")
    )
    Sys.unsetenv("GRUPOB_PARAMETROS")

    minutos <- round(as.numeric(difftime(Sys.time(), t0, units = "mins")), 1)
    cat(id, "| estado:", estado, "| minutos:", minutos, "\n")

    # registro de tiempos (para el criterio de costo computacional)
    fwrite(data.table(id = id, cfg = cfg, semilla = semilla, estado = estado,
                      minutos = minutos, fin = format(Sys.time())),
           ARCHIVO_TIEMPOS, append = file.exists(ARCHIVO_TIEMPOS))

    fallos <- if (identical(as.integer(estado), 0L)) 0 else fallos + 1
    if (fallos >= 2) stop("2 fallos consecutivos. Revisar errores.log en ", carpeta)
  }
}
cat("\nFINALIZO\n")

Ya completada: E5_S770503 

INICIANDO: E5_S216751 2026-11-02 16:26:46 
E5_S216751 | estado: 0 | minutos: 16.8 

INICIANDO: E5_S126233 2026-11-02 16:43:37 
E5_S126233 | estado: 0 | minutos: 17.2 

INICIANDO: E5_S877577 2026-11-02 17:00:50 
E5_S877577 | estado: 0 | minutos: 17.1 

INICIANDO: E5_S388403 2026-11-02 17:17:59 
E5_S388403 | estado: 0 | minutos: 16.8 

INICIANDO: E5_S308563 2026-11-02 17:34:46 
E5_S308563 | estado: 0 | minutos: 16.8 

INICIANDO: E5_S344785 2026-11-02 17:51:32 
E5_S344785 | estado: 0 | minutos: 16.4 

INICIANDO: E5_S618973 2026-11-02 18:07:56 
E5_S618973 | estado: 0 | minutos: 16.4 

INICIANDO: E5_S245039 2026-11-02 18:24:20 
E5_S245039 | estado: 0 | minutos: 16.5 

INICIANDO: E5_S552707 2026-11-02 18:40:53 
E5_S552707 | estado: 0 | minutos: 15.9 

FINALIZO


In [11]:
# ==========================================
# CONSOLIDAR RESULTADOS Y WILCOXON
# ==========================================

fs <- list.files(BASE, pattern = "^PARAM\\.yml$", recursive = TRUE, full.names = TRUE)

tb <- rbindlist(lapply(fs, function(f) {
  p <- read_yaml(f)
  data.table(
    id      = p$id_ejecucion,
    cfg     = p$configuracion,
    semilla = p$semilla_primigenia,
    gan     = p$resultado$ganancia_suavizada_max,
    envios  = p$resultado$envios
  )
}))

# tiempos: se toma el ultimo registro exitoso de cada corrida
if (file.exists(ARCHIVO_TIEMPOS)) {
  tt <- fread(ARCHIVO_TIEMPOS)[estado == 0, .(minutos = tail(minutos, 1)), by = id]
  tb <- merge(tb, tt, by = "id", all.x = TRUE)
} else tb[, minutos := NA_real_]

setorder(tb, cfg, semilla)
fwrite(tb, file.path(BASE, "consolidado_resultados.csv"))
print(tb)

print(tb[, .(n = .N, gan_media = mean(gan), gan_sd = sd(gan),
             min_medio = mean(minutos, na.rm = TRUE)), by = cfg])

# Wilcoxon pareado por semilla contra E0, con ajuste de Holm
w <- dcast(tb, semilla ~ cfg, value.var = "gan")
otras <- intersect(c("E1", "E2", "E3", "E4" , "E5"), names(w))

if ("E0" %in% names(w) && length(otras) > 0) {
  n_pares <- sapply(otras, function(e) sum(complete.cases(w[, c("E0", e), with = FALSE])))
  pv <- sapply(otras, function(e) {
    ok <- complete.cases(w[, c("E0", e), with = FALSE])
    if (sum(ok) < 6) return(NA_real_)          # muy pocos pares para concluir
    wilcox.test(w[[e]][ok], w$E0[ok], paired = TRUE)$p.value
  })
  print(data.table(config = otras, n_pares = n_pares,
                   p_crudo = pv, p_holm = p.adjust(pv, method = "holm")))
}

Key: <id>
            id    cfg semilla     gan envios minutos
        <char> <char>   <num>   <num>  <int>   <num>
 1: E5_S126233     E5  126233 89.5900   1643    17.2
 2: E5_S216751     E5  216751 90.9125   1966    16.8
 3: E5_S245039     E5  245039 90.9275   2401    16.5
 4: E5_S308563     E5  308563 91.1700   1990    16.8
 5: E5_S344785     E5  344785 90.8975   1825    16.4
 6: E5_S388403     E5  388403 87.7900   2270    16.8
 7: E5_S552707     E5  552707 92.2900   1832    15.9
 8: E5_S618973     E5  618973 90.1450   1979    16.4
 9: E5_S770503     E5  770503 91.4350   1763    17.5
10: E5_S877577     E5  877577 90.2375   1780    17.1
      cfg     n gan_media  gan_sd min_medio
   <char> <int>     <num>   <num>     <num>
1:     E5    10   90.5395 1.21973     16.74
